# Learn-a-World v3: Phase 3 T1 Joint Retrain (Repair Protocol)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sahiltolani30/learn-a-world/blob/main/notebooks/05_phase3_t1_joint_repair.ipynb)

**Campaign Context & Core Directive:**
- **Ground Truth**: Google Drive is the sole source of truth for all data, manifests, and checkpoints.
- **Root Cause Remediation**: Curiosity-only retraining caused policy drift on combat (`defeat_zombie`: 20% -> 0%). This notebook implements **Joint Retraining** on all four audited data families:
  1. `structured_random` (ONLY `_final` cumulative snapshot: ~150k transitions)
  2. `teacher_hybrid` (ONLY `_final` cumulative snapshot: ~150k transitions)
  3. `planner_v2` (ONLY `_final` cumulative snapshot: ~30k transitions)
  4. `curiosity_v3` (ALL disjoint exploration shards: ~160k transitions)
- **Total Joint Buffer**: ~480,000 – 490,000 transitions.
- **Starting Checkpoint**: Strictly `checkpoints/jepa_crafter_v3_phase2.pt` (pre-Phase-3 baseline).
- **Target Bar**: Crafter Score > **2.313%** AND `defeat_zombie` > **0.0%** (Real target: **2.74%** v2 champion).

---

### Step 1: Mount Google Drive
Mounts Google Drive to access all ground-truth datasets, checkpoints, and logs.

In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/learn-a-world'
drive_p = Path(DRIVE_DIR)
(drive_p / 'checkpoints').mkdir(parents=True, exist_ok=True)
(drive_p / 'results').mkdir(parents=True, exist_ok=True)
(drive_p / 'data' / 'combined_shards_run3').mkdir(parents=True, exist_ok=True)

print('Google Drive connected at:', DRIVE_DIR)

### Step 2: Clone or Pull Repository on Branch `main` & Install Requirements

In [ ]:
%%bash
cd /content
if [ ! -d "learn-a-world" ]; then
  git clone https://github.com/sahiltolani30/learn-a-world.git
fi
cd learn-a-world
git fetch origin
git checkout main
git pull origin main
echo "--> Latest Git Commit:"
git log -1 --oneline

pip install -q -r requirements.txt
pip install -q crafter gymnasium pytest
nvidia-smi

### Step 3: Run Pre-Flight Tests
Validates that all architecture and collector unit tests pass.

In [ ]:
!cd /content/learn-a-world && python3 -m pytest tests/test_phase3_curiosity.py -v

### Step 4: Step 0 & 1 Audits on Drive (Accounting & Manifest)
Inspects Drive logs directly:
1. Accounts for tonight's run in `step3_t1_train_results.json`.
2. Recounts curiosity shards (0000-0069 vs 0075) and exact `defeat_zombie` events.

In [ ]:
import sys
sys.path.insert(0, '/content/learn-a-world')
from training.run_v3_colab_run3 import audit_tonight_run, audit_curiosity_manifest

# 1. Account for tonight's run
train_audit = audit_tonight_run(DRIVE_DIR)

# 2. Audit curiosity shards & recount defeat_zombie
curiosity_audit = audit_curiosity_manifest(DRIVE_DIR)

### Step 5: Step 2 & 3 Drive Base Shard Discovery & Joint Buffer Verification
Finds the 3 cumulative `_final` base shards directly on Drive and runs a dry-run buffer build.
- Structured Random: ONLY `_final` (~150,000 transitions)
- Teacher Hybrid: ONLY `_final` (~150,000 transitions)
- Planner v2: ONLY `_final` (~30,000 transitions)
- Curiosity Shards: ALL .npz files (~160,000 transitions)

**Hard Assertion**: Total transitions must be $\ge 450,000$.

In [ ]:
from training.run_v3_colab_run3 import find_drive_base_shards, build_joint_buffer

# 1. Locate the 3 ground-truth base final shards on Drive
base_shards = find_drive_base_shards(DRIVE_DIR)

# 2. Dry-run joint buffer load with strict validation
buffer, composition = build_joint_buffer(
    structured_random_final=base_shards['structured_random_final'],
    teacher_hybrid_final=base_shards['teacher_hybrid_final'],
    planner_v2_final=base_shards['planner_v2_final'],
    curiosity_dir=f'{DRIVE_DIR}/data/curiosity_shards',
)

print('Buffer verification passed! Total transitions:', composition['total_transitions'])

### Step 6: Step 4 — Joint Retraining (15,000 Steps)
Trains JEPA v3 on the audited joint buffer:
- **Base Checkpoint**: Strictly `jepa_crafter_v3_phase2.pt` (never from T1 curiosity-only model).
- **Sampling**: `event_balanced` (preserves combat & rare crafting events).
- **Output**: `checkpoints/jepa_crafter_v3_t1_joint.pt`.

In [ ]:
from training.run_v3_colab_run3 import retrain_t1_joint

ckpt_path, buffer_composition = retrain_t1_joint(
    drive_dir=DRIVE_DIR,
    structured_random_final=base_shards['structured_random_final'],
    teacher_hybrid_final=base_shards['teacher_hybrid_final'],
    planner_v2_final=base_shards['planner_v2_final'],
    curiosity_dir=f'{DRIVE_DIR}/data/curiosity_shards',
    base_checkpoint=f'{DRIVE_DIR}/checkpoints/jepa_crafter_v3_phase2.pt',
    num_steps=15_000,
    batch_size=16,
    seq_len=16,
    lr=2e-4,
    device='cuda',
)

print('Joint retrain complete! Output checkpoint:', ckpt_path)

### Step 7: Step 5 — Screening Evaluation (50 Episodes, Seeds 2050–2099)
Evaluates the jointly retrained model on the untouched 50-episode test seeds.
Results are saved to `results/screen_v3_t1_joint_results.json` with embedded buffer composition.

In [ ]:
from training.run_v3_colab_run3 import screen_t1_joint

results = screen_t1_joint(
    checkpoint_path=ckpt_path,
    drive_dir=DRIVE_DIR,
    buffer_composition=buffer_composition,
    num_episodes=50,
    seed_start=2050,
    device='cuda',
)

### Step 8: Gate Check & Results Push to GitHub
Evaluates whether the checkpoint clears the T1 Gate:
- Crafter Score > 2.313%
- `defeat_zombie` > 0.0%

Commits only the JSON screening results to GitHub (keeps large binaries on Drive).

In [ ]:
from training.run_v3_colab_run3 import gate_check_t1_joint

# Evaluate gate
gate_passed = gate_check_t1_joint(results)

# Commit screening results to GitHub
!cd /content/learn-a-world && \
 git config --global user.name "Sahil Tolani" && \
 git config --global user.email "sahiltolani30@gmail.com" && \
 git add results/screen_v3_t1_joint_results.json && \
 git commit -m "results(phase3): record T1 joint retrain screening results" && \
 git push origin main